# FHIR-Aggregator: A Catalog of Research Data
The FHIR Aggregator acts as a centralized repository for diverse healthcare data, organized using the FHIR (Fast Healthcare Interoperability Resources) standard. It provides researchers access to a wide range of information, including:

* Clinical data: Patient demographics, conditions, medications, observations, and procedures.
* Research studies: Information about research projects, participants, and study protocols.
* OMICS data associated with Specimens

## Overview  

This notebook leverages **[FHIR GraphDefinition](https://hl7.org/fhir/graphdefinition.html)** objects to define and execute graph-based traversals across multiple interconnected FHIR resource graphs. The data retrieved is written to a **local SQLite database** for persistence and later transformed into **analyst-friendly dataframes** for analysis using tools like Python’s pandas library.


### Motivation  

**[FHIR Search](https://www.hl7.org/fhir/search.html)** provides a robust querying framework but comes with significant limitations:  

1. **Deep Chaining Limits**:  
   Chaining searches (e.g., `Patient -> Observation -> Encounter -> Procedure`) often hits server depth limitations.  

2. **Inefficient Query Execution**:  
   Searching deeply related resources requires multiple chained requests, leading to performance issues and unnecessary round trips.  

3. **Lack of Explicit Traversals**:  
   Relationships in FHIR are implicit in references (e.g., `Observation.subject` pointing to `Patient`). This implicit structure requires manual composition of queries, which is prone to errors.  

4. **Differences between servers**:  
   Relationships in FHIR are implicit in references (e.g., `Observation.subject` pointing to `Patient`). This implicit structure requires manual composition of queries, which is prone to errors. **Key differences:**

    1. Supported Search Parameters:

    The FHIR specification defines a core set of search parameters for each resource type. However, servers might implement only a subset of these parameters or extend them with custom ones. This means a search that works on one server might not be valid on another.
    2. Chaining and Search Depth:

    Servers have different limits on how deeply you can chain search parameters (e.g., searching for Patients with Observations linked to specific Encounters). Some servers might restrict chaining depth, impacting the complexity of queries you can execute.
    3. Search Parameter Behavior:

    Even for commonly supported parameters, the specific behavior might differ. For instance, how date ranges are handled or how string searches are performed can vary, leading to unexpected results when switching between servers.
    4. Terminology Services:

    Servers might use different terminology services for coding systems, impacting how searches using codes are resolved. This can lead to discrepancies in results if the terminology is not aligned.
    5. Performance and Optimization:

    Search performance can vary significantly due to server infrastructure, indexing strategies, and data volume. Certain search patterns might be optimized on one server but inefficient on another.
    6. Data Models and Link Population:

    Data Models: FHIR servers may implement different profiles or extensions on top of the base FHIR specification. This can affect the structure of resources and the availability of certain data elements relevant to research.
    Link Population: The extent to which servers populate links (references) between resources can vary. Some servers might aggressively pre-fetch linked resources, while others may require explicit chaining or separate requests to retrieve them. This can significantly impact search performance and the complexity of queries needed to access related data for analysis.   




## Solution

#### fq (fhir-query): Your FHIR Querying Assistant

By using **FHIR GraphDefinition**, we declaratively define resource relationships and efficiently retrieve data. Once retrieved, the data is stored locally and can be transformed into dataframes for advanced analysis.


The [fhir-aggregator-client](https://github.com/FHIR-Aggregator/fhir-aggregator-client) tool runs an R5 GraphDefinition against a FHIR server.  



### Key Features  

- **GraphDefinition-Driven Traversals**: Use GraphDefinition objects to define explicit relationships between resources and automate traversal logic.  
- **Local SQLite Storage**: Persist the retrieved FHIR data in a local SQLite database for querying and offline analysis.  
- **Analyst-Friendly Dataframes**: Convert stored FHIR resources into pandas dataframes for ease of use in analytical workflows.  
- **Reusable Graph Definitions**: Maintain a library of GraphDefinition YAML files that can be reused across different workflows and projects.  Researchers and Data submitters can publish GraphDefinition files to help others navigate their data.




### Installation

The fq utility, short for "fhir-query," is a command-line tool specifically designed to simplify the process of interacting with FHIR servers. It provides researchers with a convenient way to:

1. Retrieve the vocabulary of a FHIR server: With the vocabulary command, fq fetches and summarizes the key data elements (CodeableConcepts and Extensions) used within the FHIR data. This creates a central vocabulary Dataframe that helps researchers identify important data elements and their usage within the server.


2. Execute queries to retrieve FHIR resources: Researchers can then use fq to execute FHIR queries using a readable syntax. This helps to retrieve and filter data from the FHIR Server based on various search parameters and criteria.

In [ ]:
%%capture
!pip install fhir-aggregator-client --no-cache-dir --quiet
!pip freeze | grep fhir_aggregator_client

#### Verify the tools were installed

In [ ]:
!fq

### Useage

#### List the installed GraphDefinition files


In [ ]:
!fq ls

#### Run a GraphDefiniton

In [ ]:
%env FHIR_BASE=https://google-fhir.fhir-aggregator.org


In [ ]:
!fq run cholangiocarcinoma-graph '/Condition?code:code=70179006'

In [ ]:
# !fq run  condition-graph '/Condition?code:text=cholangiocarcinoma'

#### Analyse Results

The graph represents relationships between different FHIR resources.Examples of FHIR resources include Patient, Condition, Observation, Procedure, etc.

Each node is labled as: <resource_type\>/<count\> the number of records of that type retrieved.

The edges in the graph are weighted.  The thicker the line, the more connections there are between nodes.


In [ ]:
# Create a graph of the results
!fq results visualize

In [ ]:
# Read the locally stored HTML file containing a graph visualization and displaying it within the Jupyter notebook.

from IPython.display import HTML
with open('fhir-graph.html', 'r') as file:
    html_content = file.read()

# Set the display height (in pixels)
display(HTML("<div style='height: 800px;'>{}</div>".format(html_content)))


#### Create a dataframe of results

In [ ]:
!fq results dataframe

In [ ]:

import pandas as pd

df = pd.read_csv('fhir-graph.tsv', sep='\t')

df


#### Other servers

You can use the `fq` tool with other FHIR servers.  For example, this query retrieves a study from `dbGAP`

In [ ]:
# delete the previous results, start with a fresh database
!rm ~/.fhir-aggregator/fhir-graph.sqlite
!fq run  --fhir-base-url https://dbgap-api.ncbi.nlm.nih.gov/fhir-jpa-pilot/x1  research-study-link-iterate  '/ResearchStudy?_id=phs001232'


In [ ]:
# use the same commands to analyse results
!fq results visualize

In [ ]:
# create a graph of the results

from IPython.display import HTML
with open('fhir-graph.html', 'r') as file:
    html_content = file.read()

# Set the display height (in pixels)
display(HTML("<div style='height: 800px;'>{}</div>".format(html_content)))

In [ ]:
# create a dataframe of results
!fq results dataframe

In [ ]:
import pandas as pd

df = pd.read_csv('fhir-graph.tsv', sep='\t')

df
